In [ ]:
#JSON output parser
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import (
    JsonOutputParser,
    StructuredOutputParser,
    ResponseSchema,
)


load_dotenv()

model = ChatOpenAI()
parser = JsonOutputParser()

template1 = PromptTemplate(
    template="Write me some facts about{topic} \n{format_instructions}",
    input_variables=["topic"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = template1 | model | parser

result = chain.invoke({"topic": "Badminton Racquet"})
print(result)


# --> Structured output parser
schema = [
    ResponseSchema(name="fact_1", description="Fact 1 about the topic"),
    ResponseSchema(name="fact_2", description="Fact 2 about the topic"),
    ResponseSchema(name="fact_3", description="Fact 3 about the topic"),
]

parser = StructuredOutputParser.from_response_schemas(schema)

template1 = PromptTemplate(
    template="Write me some facts about{topic} \n{format_instructions}",
    input_variables=["topic"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = template1 | model | parser

result = chain.invoke({"topic": "Badminton Racquet"})
print(result)


In [ ]:
#pydantic output parser
class person(BaseModel):
    name: str = Field(description=" The name of the person")
    age: int = Field(gt=15, description="The age of the person")
    city: str = Field(description="The name of the city it belongs to")


parser = PydanticOutputParser(pydantic_object=person)
template = PromptTemplate(
    template="Generate me the name, age , city of {anime} worls\n {format_instruction}",
    input_variables=["anime"],
    partial_variables={"format_instruction": parser.get_format_instructions()},
)

chain = template | model | parser

result = chain.invoke({"anime": "Black clover"})
print(result)

In [ ]:
#string output parser
template1 = PromptTemplate(
    template="Write me 5 lines on {topic}", input_variables=["topic"]
)

template2 = PromptTemplate(
    template="write me 3 important bullet points{text}", input_variables=["text"]
)

parser = StrOutputParser()

chain = template1 | model | parser | template2 | model | parser

result = chain.invoke({"topic": "Langchain"})
print(result)

In [ ]:
# With_ouput_structure_LLM

from dotenv import load_dotenv
from typing import Optional, Literal
from pydantic import BaseModel, Field
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task="text-generation",  # show error because it doesn't support with_structure_output we need Output parser
)

model = ChatHuggingFace(llm=llm)


# schema
class Review(BaseModel):
    key_themes: list[str] = Field(
        description="Write down all the key themes discussed in the review in a list"
    )
    summary: str = Field(description="A brief summary of the review")
    sentiment: Literal["pos", "neg"] = Field(
        description="Return sentiment of the review either negative, positive or neutral"
    )
    pros: Optional[list[str]] = Field(
        default=None, description="Write down all the pros inside a list"
    )
    cons: Optional[list[str]] = Field(
        default=None, description="Write down all the cons inside a list"
    )
    name: Optional[str] = Field(
        default=None, description="Write the name of the reviewer"
    )


structured_model = model.with_structured_output(Review)

result = structured_model.invoke("""he Triumph Speed 400 is a premium modern-retro bike that combines classic Triumph styling with strong performance and everyday comfort. It comes with a 398cc liquid-cooled engine producing around 40 PS power, making it smooth, refined, and fun for both city rides and highways.Key highlights include:Premium build qualityComfortable riding postureGood handling and stabilityDual-channel ABS and traction controlStylish retro-modern design ProsSmooth and powerful enginePremium look and feelEasy to handle for beginnersGood balance for city + touringConsSlight engine heat in trafficRear suspension feels stiff sometimesLimited service network compared to bigger brandsPillion comfort is average for long ridesOverall, the Speed 400 is one of the best all-rounder bikes in the 400cc segment for riders wanting a premium, stylish, and refined motorcycle. Summarize by Shiva
""")

print(result)


In [ ]:
#with_output_structure_typedict

from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Literal

load_dotenv()

model = ChatOpenAI()


class Bike(TypedDict):
    summary: Annotated[str, "A brief summary of bike model is 2026"]
    features: Annotated[list[str], "Feature of bike and aslo the colors"]
    pros: Annotated[Literal["Pos", "Neg"], "Just say Positive as 1 and Negative as 0"]
    name: Annotated[str, "Reviewd by the person"]


struct_model = model.with_structured_output(Bike)

result = struct_model.invoke(
    "The Triumph Speed 400 is a premium modern-retro bike that combines classic Triumph styling with strong performance and everyday comfort. It comes with a 398cc liquid-cooled engine producing around 40 PS power, making it smooth, refined, and fun for both city rides and highways.Key highlights include:Premium build qualityComfortable riding postureGood handling and stabilityDual-channel ABS and traction controlStylish retro-modern design ProsSmooth and powerful enginePremium look and feelEasy to handle for beginnersGood balance for city + touringConsSlight engine heat in trafficRear suspension feels stiff sometimesLimited service network compared to bigger brandsPillion comfort is average for long ridesOverall, the Speed 400 is one of the best all-rounder bikes in the 400cc segment for riders wanting a premium, stylish, and refined motorcycle. Summarize by Shiva"
)

print(result["name"])



In [ ]:
#with_output_structure_pydantic

from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import Optional, Literal
from pydantic import BaseModel, Field

load_dotenv()

model = ChatOpenAI()


# schema
class Review(BaseModel):
    key_themes: list[str] = Field(
        description="Write down all the key themes discussed in the review in a list"
    )
    summary: str = Field(description="A brief summary of the review")
    sentiment: Literal["pos", "neg"] = Field(
        description="Return sentiment of the review either negative, positive or neutral"
    )
    pros: Optional[list[str]] = Field(
        default=None, description="Write down all the pros inside a list"
    )
    cons: Optional[list[str]] = Field(
        default=None, description="Write down all the cons inside a list"
    )
    name: Optional[str] = Field(
        default=None, description="Write the name of the reviewer"
    )


structured_model = model.with_structured_output(Review)

result = structured_model.invoke("""
BMW is a premium German automobile company known for luxury,
performance, advanced technology, and sporty driving experience.
The brand manufactures luxury cars, SUVs, electric vehicles,
and high-performance M series cars.

BMW cars are popular for:
- Powerful engines
- Premium interior quality
- Modern technology features
- Excellent driving dynamics
- Stylish and sporty design

Some famous BMW series include:
- BMW 3 Series
- BMW 5 Series
- BMW 7 Series
- BMW X Series SUVs
- BMW M Performance models

Pros:
- Luxury and comfort
- Strong performance
- Advanced safety features
- Premium brand value

Cons:
- High maintenance cost
- Expensive spare parts
- Lower mileage compared to economy cars

Overall, BMW is considered one of the top luxury car brands
in the world for people who want performance, comfort,
and premium driving experience.
review by Mohit
""")

print(result)

